# BO-Guided Center Adaptation with DORAEMON (Gym surrogate -> TinySim target)

This notebook implements a practical prototype of your algorithm:
- warm-start DQN policy from existing MountainCar checkpoint
- BO ask/tell over DR center $\mu=(f,g)$
- DORAEMON-style spread adaptation over DR std $\sigma$
- small-budget target evals in TinySim each BO iteration

Target parameters are only used in **target evaluation** (blind BO setup).


## 1) Imports and Global Setup

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import time
import copy
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from tinysim.mountain_car import MountainCarEnv


In [ ]:
# BO dependency check (chosen engine: scikit-optimize GP ask/tell)
try:
    from skopt import Optimizer
    from skopt.space import Real
except Exception as exc:
    raise ImportError(
        "scikit-optimize is required. Install with: `pip install scikit-optimize`"
    ) from exc

print('skopt imported successfully')


## 2) Config (edit this cell)

- `TARGET_PARAMS` is used **only** for TinySim target evaluation.
- BO never gets direct access to target params beyond scalar target eval score.


In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

PROJECT_DIR = Path('/home/buchkeva/IdeaTesting/TinySim')
RUNS_ROOT = PROJECT_DIR / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# Warm start checkpoint from your current MountainCar DQN notebook
WARMSTART_CHECKPOINT = PROJECT_DIR / 'mountain_car_saved' / 'mountain_car_dqn.pt'

# Domain randomization bounds over (force, gravity) in Gym surrogate
# Keep DR_BOUNDS broad, and optionally use EDGE_DR_BOUNDS for hard-target runs.
DR_BOUNDS = {
    'force': (0.0003, 0.0030),
    'gravity': (0.0010, 0.0062),
}
EDGE_DR_BOUNDS = {
    'force': (0.0007, 0.0018),
    'gravity': (0.0030, 0.0059),
}

# Initial center/spread
MU0 = np.array([0.0010, 0.0025], dtype=np.float32)  # Gym default-like center [force, gravity]
MU0_EDGE = np.array([
    float(np.clip(MU0[0], EDGE_DR_BOUNDS['force'][0], EDGE_DR_BOUNDS['force'][1])),
    float(np.clip(MU0[1], EDGE_DR_BOUNDS['gravity'][0], EDGE_DR_BOUNDS['gravity'][1])),
], dtype=np.float32)
SIGMA0 = np.array([0.00018, 0.00060], dtype=np.float32)
SIGMA_MIN = np.array([0.00003, 0.00010], dtype=np.float32)
SIGMA_MAX = np.array([0.00060, 0.00150], dtype=np.float32)

# DORAEMON-style adaptation controls (hysteresis band + update interval)
alpha_low = 0.52
alpha_high = 0.78
sigma_expand = 1.08
sigma_shrink = 0.92
sigma_update_interval = 4
surrogate_mode = 'stress'  # 'stress' or 'random'

# Environment horizon / solve definition
tau_steps = 200

# BO/adaptation stabilization
bo_epsilon_reset = 0.12     # reset exploration floor at each BO step
contender_margin = 0.15     # if candidate is close to best, spend extra target eval budget
early_stop_patience = 8     # stop early when near-solved and not improving

# TinySim target (used only by target evaluator)
TARGET_PARAMS = {
    'force': 0.0010,
    'gravity': 0.0050,
}

# Budgets
SMOKE_CFG = dict(T=3, K=20, B_r=5, B_r_extra=0, surrogate_eval_episodes=6)
PROTO_CFG = dict(T=12, K=120, B_r=12, B_r_extra=8, surrogate_eval_episodes=10)
EDGE_CFG = dict(T=20, K=180, B_r=20, B_r_extra=20, surrogate_eval_episodes=16)

print('Warmstart checkpoint:', WARMSTART_CHECKPOINT)
print('Target params (eval only):', TARGET_PARAMS)
print('Surrogate mode:', surrogate_mode)
print('MU0 (default):', MU0.tolist())
print('MU0_EDGE (projected):', MU0_EDGE.tolist())

## 3) DQN Components (MountainCar)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int = 10000):
        self.capacity = int(capacity)
        self.buffer = []
        self.pos = 0

    def push(self, state, action, reward, next_state, done):
        item = (
            np.asarray(state, dtype=np.float32),
            int(action),
            float(reward),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        )
        if len(self.buffer) < self.capacity:
            self.buffer.append(item)
        else:
            self.buffer[self.pos] = item
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int, device: str):
        idx = np.random.choice(len(self.buffer), size=batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        s, a, r, ns, d = zip(*batch)
        states = torch.tensor(np.stack(s), dtype=torch.float32, device=device)
        actions = torch.tensor(a, dtype=torch.long, device=device)
        rewards = torch.tensor(r, dtype=torch.float32, device=device)
        next_states = torch.tensor(np.stack(ns), dtype=torch.float32, device=device)
        dones = torch.tensor(d, dtype=torch.float32, device=device)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


class QNetwork(nn.Module):
    def __init__(self, state_dim: int = 2, n_actions: int = 3, hidden_dim: int = 128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


@dataclass
class DQNConfig:
    state_dim: int = 2
    n_actions: int = 3
    hidden_dim: int = 128
    lr: float = 1e-3
    gamma: float = 0.99
    epsilon_start: float = 1.0
    epsilon_end: float = 0.01
    epsilon_decay: float = 0.995
    buffer_capacity: int = 10000
    batch_size: int = 64
    target_update_freq: int = 10


class DQNAgent:
    def __init__(self, cfg: DQNConfig, device: str = 'cpu'):
        self.cfg = cfg
        self.device = device

        self.policy_net = QNetwork(cfg.state_dim, cfg.n_actions, cfg.hidden_dim).to(device)
        self.target_net = QNetwork(cfg.state_dim, cfg.n_actions, cfg.hidden_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg.lr)
        self.memory = ReplayBuffer(cfg.buffer_capacity)

        self.epsilon = cfg.epsilon_start
        self.training_losses = []
        self._episode_counter = 0

    def act(self, state: np.ndarray, training: bool = True) -> int:
        if training and random.random() < self.epsilon:
            return random.randrange(self.cfg.n_actions)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            q = self.policy_net(s)
            return int(q.argmax(dim=1).item())

    def store(self, s, a, r, ns, done):
        self.memory.push(s, a, r, ns, done)

    def learn_step(self):
        if len(self.memory) < self.cfg.batch_size:
            return None
        s, a, r, ns, d = self.memory.sample(self.cfg.batch_size, self.device)

        q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            nq = self.target_net(ns).max(dim=1)[0]
            target = r + self.cfg.gamma * nq * (1.0 - d)

        loss = nn.MSELoss()(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.training_losses.append(float(loss.item()))
        return float(loss.item())

    def end_episode(self):
        self._episode_counter += 1
        self.epsilon = max(self.cfg.epsilon_end, self.epsilon * self.cfg.epsilon_decay)
        if (self._episode_counter % self.cfg.target_update_freq) == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

    def state_payload(self) -> dict[str, Any]:
        return {
            'policy_state_dict': self.policy_net.state_dict(),
            'target_state_dict': self.target_net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'epsilon': self.epsilon,
            'training_losses': self.training_losses,
            'hparams': vars(self.cfg),
        }

    def load_payload(self, payload: dict[str, Any]):
        self.policy_net.load_state_dict(payload['policy_state_dict'])
        self.target_net.load_state_dict(payload.get('target_state_dict', payload['policy_state_dict']))
        if 'optimizer_state_dict' in payload:
            self.optimizer.load_state_dict(payload['optimizer_state_dict'])
        self.epsilon = float(payload.get('epsilon', self.cfg.epsilon_end))
        self.training_losses = list(payload.get('training_losses', []))


## 4) Warm-start and Checkpoint Helpers

In [ ]:
def make_agent_from_hparams(hparams: dict | None, device: str = DEVICE) -> DQNAgent:
    if hparams is None:
        cfg = DQNConfig()
    else:
        cfg = DQNConfig(**{k: v for k, v in hparams.items() if k in DQNConfig.__annotations__})
    return DQNAgent(cfg=cfg, device=device)


def load_warmstart_agent(checkpoint_path: str | Path, device: str = DEVICE) -> DQNAgent:
    payload = torch.load(str(checkpoint_path), map_location=device)
    hparams = payload.get('hparams')
    agent = make_agent_from_hparams(hparams, device=device)
    agent.load_payload(payload)
    return agent


def save_agent_checkpoint(agent: DQNAgent, path: str | Path, extra: dict | None = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = agent.state_payload()
    payload['extra'] = extra or {}
    torch.save(payload, str(path))


def clone_agent(agent: DQNAgent) -> DQNAgent:
    cloned = make_agent_from_hparams(vars(agent.cfg), device=agent.device)
    cloned.load_payload(agent.state_payload())
    return cloned


## 5) Environment and Evaluation Helpers

In [ ]:
def make_gym_env(force: float, gravity: float, seed: int | None = None):
    env = gym.make('MountainCar-v0')
    env.reset(seed=seed)
    env.unwrapped.force = float(force)
    env.unwrapped.gravity = float(gravity)
    return env


def clip_params(force: float, gravity: float, bounds: dict) -> tuple[float, float]:
    f = float(np.clip(force, bounds['force'][0], bounds['force'][1]))
    g = float(np.clip(gravity, bounds['gravity'][0], bounds['gravity'][1]))
    return f, g


def sample_dr_params(mu: np.ndarray, sigma: np.ndarray, bounds: dict, rng: np.random.Generator):
    # independent Gaussian over [force, gravity] + clipping
    force = rng.normal(mu[0], sigma[0])
    gravity = rng.normal(mu[1], sigma[1])
    return clip_params(force, gravity, bounds)


def build_stress_eval_params(
    mu: np.ndarray,
    sigma: np.ndarray,
    bounds: dict,
    n_eval: int,
    rng: np.random.Generator,
) -> list[tuple[float, float]]:
    """
    Stress-oriented surrogate probes near mu.
    Includes center, hard corners, and fills the remainder with random DR samples.
    """
    k_hi = 1.5
    k_mid = 1.0

    pts = [
        clip_params(mu[0], mu[1], bounds),
        clip_params(mu[0] - k_hi * sigma[0], mu[1] + k_hi * sigma[1], bounds),  # harder
        clip_params(mu[0] - k_mid * sigma[0], mu[1] + k_mid * sigma[1], bounds),
        clip_params(mu[0] + k_mid * sigma[0], mu[1] - k_mid * sigma[1], bounds),  # easier
        clip_params(mu[0] + k_hi * sigma[0], mu[1] - k_hi * sigma[1], bounds),
    ]

    if n_eval <= len(pts):
        return pts[:n_eval]

    while len(pts) < n_eval:
        pts.append(sample_dr_params(mu, sigma, bounds, rng))
    return pts


def expand_entropy(sigma: np.ndarray, sigma_max: np.ndarray, factor: float = 1.10) -> np.ndarray:
    return np.minimum(sigma * factor, sigma_max)


def shrink_or_hold(sigma: np.ndarray, sigma_min: np.ndarray, factor: float = 0.90) -> np.ndarray:
    return np.maximum(sigma * factor, sigma_min)


def surrogate_success_rate(
    agent: DQNAgent,
    mu: np.ndarray,
    sigma: np.ndarray,
    n_eval: int,
    tau_steps: int,
    bounds: dict,
    seed: int,
    mode: str = 'stress',
) -> float:
    rng = np.random.default_rng(seed)
    successes = 0

    if mode == 'stress':
        eval_params = build_stress_eval_params(mu, sigma, bounds, n_eval=n_eval, rng=rng)
    else:
        eval_params = [sample_dr_params(mu, sigma, bounds, rng) for _ in range(n_eval)]

    for i, (force, gravity) in enumerate(eval_params):
        env = make_gym_env(force=force, gravity=gravity, seed=seed + i)
        s, _ = env.reset(seed=seed + i)

        solved = False
        for _ in range(tau_steps):
            a = agent.act(s, training=False)
            ns, _, terminated, truncated, _ = env.step(a)
            s = ns
            if terminated:
                solved = True
                break
            if truncated:
                break

        env.close()
        successes += int(solved)

    return float(successes / max(1, n_eval))


def target_eval_tinysim_detailed(
    agent: DQNAgent,
    n_eval_target: int,
    max_steps: int,
    target_params: dict,
    seed: int,
) -> tuple[float, int, int]:
    successes = 0

    for ep in range(n_eval_target):
        sim_env = MountainCarEnv()
        sim_env.force = float(target_params['force'])
        sim_env.gravity = float(target_params['gravity'])

        state = sim_env.reset()
        obs = np.array([state['position'], state['velocity']], dtype=np.float32)

        solved = False
        for _ in range(max_steps):
            a = agent.act(obs, training=False)
            state = sim_env.step(a)
            obs = np.array([state['position'], state['velocity']], dtype=np.float32)
            if bool(state['done']):
                solved = True
                break

        successes += int(solved)

    rate = float(successes / max(1, n_eval_target))
    return rate, int(successes), int(n_eval_target)


def target_eval_tinysim(
    agent: DQNAgent,
    n_eval_target: int,
    max_steps: int,
    target_params: dict,
    seed: int,
) -> float:
    rate, _, _ = target_eval_tinysim_detailed(
        agent=agent,
        n_eval_target=n_eval_target,
        max_steps=max_steps,
        target_params=target_params,
        seed=seed,
    )
    return float(rate)

## 6) DORAEMON Adaptation Under DR

In [ ]:
def train_under_dr(
    agent: DQNAgent,
    mu: np.ndarray,
    sigma_init: np.ndarray,
    K: int,
    bounds: dict,
    tau_steps: int,
    alpha_low: float,
    alpha_high: float,
    sigma_min: np.ndarray,
    sigma_max: np.ndarray,
    surrogate_eval_episodes: int,
    sigma_expand_factor: float,
    sigma_shrink_factor: float,
    sigma_update_interval: int,
    surrogate_mode: str,
    seed: int,
):
    rng = np.random.default_rng(seed)
    sigma = sigma_init.copy().astype(np.float32)

    g_history = []
    sigma_history = []
    loss_history = []

    for k in range(K):
        force_k, gravity_k = sample_dr_params(mu, sigma, bounds, rng)
        env = make_gym_env(force=force_k, gravity=gravity_k, seed=seed + 1000 + k)

        s, _ = env.reset(seed=seed + 1000 + k)
        ep_losses = []

        for _ in range(tau_steps):
            a = agent.act(s, training=True)
            ns, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            agent.store(s, a, r, ns, float(done))
            loss = agent.learn_step()
            if loss is not None:
                ep_losses.append(loss)
            s = ns
            if done:
                break

        env.close()
        agent.end_episode()

        g_t = surrogate_success_rate(
            agent=agent,
            mu=mu,
            sigma=sigma,
            n_eval=surrogate_eval_episodes,
            tau_steps=tau_steps,
            bounds=bounds,
            seed=seed + 5000 + k,
            mode=surrogate_mode,
        )

        g_history.append(float(g_t))
        loss_history.append(float(np.mean(ep_losses)) if len(ep_losses) else np.nan)

        # Hysteresis + interval-based sigma update for stability.
        if ((k + 1) % max(1, sigma_update_interval)) == 0:
            g_window = g_history[-max(1, sigma_update_interval):]
            g_bar = float(np.mean(g_window))
            if g_bar >= alpha_high:
                sigma = expand_entropy(sigma, sigma_max=sigma_max, factor=sigma_expand_factor)
            elif g_bar <= alpha_low:
                sigma = shrink_or_hold(sigma, sigma_min=sigma_min, factor=sigma_shrink_factor)

        sigma_history.append(sigma.copy())

    return agent, g_history, sigma_history, sigma

## 7) BO + DORAEMON Main Loop

In [ ]:
def project_mu_to_bounds(mu: np.ndarray, bounds: dict) -> np.ndarray:
    return np.array([
        float(np.clip(mu[0], bounds['force'][0], bounds['force'][1])),
        float(np.clip(mu[1], bounds['gravity'][0], bounds['gravity'][1])),
    ], dtype=np.float32)


def run_bo_doraemon(
    checkpoint_path: str | Path,
    mu0: np.ndarray,
    sigma0: np.ndarray,
    T: int,
    K: int,
    B_r: int,
    bounds: dict,
    tau_steps: int,
    alpha_low: float,
    alpha_high: float,
    sigma_min: np.ndarray,
    sigma_max: np.ndarray,
    surrogate_eval_episodes: int,
    sigma_expand_factor: float,
    sigma_shrink_factor: float,
    target_params: dict,
    seed: int = 42,
    sigma_update_interval: int = 4,
    surrogate_mode: str = 'stress',
    bo_epsilon_reset: float = 0.12,
    B_r_extra: int = 0,
    contender_margin: float = 0.15,
    early_stop_patience: int | None = None,
):
    run_name = time.strftime('bo_doraemon_%Y%m%d_%H%M%S')
    run_dir = RUNS_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    hist_csv = run_dir / 'history.csv'
    hist_json = run_dir / 'history.json'
    best_ckpt = run_dir / 'best_policy.pt'
    last_ckpt = run_dir / 'last_policy.pt'

    agent_prev = load_warmstart_agent(checkpoint_path=checkpoint_path, device=DEVICE)
    sigma_prev = sigma0.copy().astype(np.float32)

    opt = Optimizer(
        dimensions=[
            Real(bounds['force'][0], bounds['force'][1], name='force_center'),
            Real(bounds['gravity'][0], bounds['gravity'][1], name='gravity_center'),
        ],
        base_estimator='GP',
        acq_func='EI',
        random_state=seed,
    )

    history = []

    mu0 = np.array(mu0, dtype=np.float32)
    mu0_proj = project_mu_to_bounds(mu0, bounds)
    if not np.allclose(mu0, mu0_proj):
        print(f"[BO init] mu0 projected from ({mu0[0]:.6f}, {mu0[1]:.6f}) to ({mu0_proj[0]:.6f}, {mu0_proj[1]:.6f})")

    # Initialize BO with warm-start center at projected mu0
    f0, succ0, eps0 = target_eval_tinysim_detailed(
        agent=agent_prev,
        n_eval_target=B_r,
        max_steps=tau_steps,
        target_params=target_params,
        seed=seed + 7,
    )
    opt.tell(mu0_proj.tolist(), -f0)

    best_score = float(f0)
    best_center = mu0_proj.copy()
    best_payload = copy.deepcopy(agent_prev.state_payload())
    save_agent_checkpoint(agent_prev, best_ckpt, extra={'center': mu0_proj.tolist(), 'score': float(f0), 'iter': 0})

    init_row = {
        'iter': 0,
        'mu_force': float(mu0_proj[0]),
        'mu_gravity': float(mu0_proj[1]),
        'sigma_force_start': float(sigma_prev[0]),
        'sigma_gravity_start': float(sigma_prev[1]),
        'sigma_force_end': float(sigma_prev[0]),
        'sigma_gravity_end': float(sigma_prev[1]),
        'g_mean': np.nan,
        'g_last': np.nan,
        'target_solve_rate': float(f0),
        'target_successes': int(succ0),
        'target_episodes': int(eps0),
        'is_best': True,
    }
    history.append(init_row)

    no_improve = 0
    final_iter = 0

    for t in range(1, T + 1):
        final_iter = t
        mu_t = np.array(opt.ask(), dtype=np.float32)

        agent_t = clone_agent(agent_prev)
        sigma_start = sigma_prev.copy()

        if bo_epsilon_reset is not None:
            agent_t.epsilon = float(max(float(agent_t.epsilon), float(bo_epsilon_reset)))

        agent_t, g_hist, sigma_hist, sigma_end = train_under_dr(
            agent=agent_t,
            mu=mu_t,
            sigma_init=sigma_start,
            K=K,
            bounds=bounds,
            tau_steps=tau_steps,
            alpha_low=alpha_low,
            alpha_high=alpha_high,
            sigma_min=sigma_min,
            sigma_max=sigma_max,
            surrogate_eval_episodes=surrogate_eval_episodes,
            sigma_expand_factor=sigma_expand_factor,
            sigma_shrink_factor=sigma_shrink_factor,
            sigma_update_interval=sigma_update_interval,
            surrogate_mode=surrogate_mode,
            seed=seed + 10000 * t,
        )

        f_base, succ_base, eps_base = target_eval_tinysim_detailed(
            agent=agent_t,
            n_eval_target=B_r,
            max_steps=tau_steps,
            target_params=target_params,
            seed=seed + 333 + t,
        )

        total_succ = int(succ_base)
        total_eps = int(eps_base)
        if int(B_r_extra) > 0 and float(f_base) >= max(0.0, float(best_score) - float(contender_margin)):
            _, succ_extra, eps_extra = target_eval_tinysim_detailed(
                agent=agent_t,
                n_eval_target=int(B_r_extra),
                max_steps=tau_steps,
                target_params=target_params,
                seed=seed + 777 + t,
            )
            total_succ += int(succ_extra)
            total_eps += int(eps_extra)

        f_t = float(total_succ / max(1, total_eps))
        opt.tell(mu_t.tolist(), -f_t)

        is_best = False
        if float(f_t) > best_score:
            best_score = float(f_t)
            best_center = mu_t.copy()
            best_payload = copy.deepcopy(agent_t.state_payload())
            save_agent_checkpoint(
                agent_t,
                best_ckpt,
                extra={'center': mu_t.tolist(), 'score': float(f_t), 'iter': t},
            )
            is_best = True
            no_improve = 0
        else:
            no_improve += 1

        row = {
            'iter': t,
            'mu_force': float(mu_t[0]),
            'mu_gravity': float(mu_t[1]),
            'sigma_force_start': float(sigma_start[0]),
            'sigma_gravity_start': float(sigma_start[1]),
            'sigma_force_end': float(sigma_end[0]),
            'sigma_gravity_end': float(sigma_end[1]),
            'g_mean': float(np.mean(g_hist)) if len(g_hist) else np.nan,
            'g_last': float(g_hist[-1]) if len(g_hist) else np.nan,
            'target_solve_rate': float(f_t),
            'target_successes': int(total_succ),
            'target_episodes': int(total_eps),
            'is_best': bool(is_best),
        }
        history.append(row)

        with open(hist_json, 'w') as f:
            json.dump(history, f, indent=2)

        with open(hist_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            writer.writeheader()
            writer.writerows(history)

        agent_prev = agent_t
        sigma_prev = sigma_end.copy()

        print(
            f"[BO step {t:02d}/{T}] mu=({mu_t[0]:.6f}, {mu_t[1]:.6f}) "
            f"g_last={row['g_last']:.3f} f_target={f_t:.3f} "
            f"target={total_succ}/{total_eps} sigma=({sigma_end[0]:.6f}, {sigma_end[1]:.6f})"
        )

        if early_stop_patience is not None and no_improve >= int(early_stop_patience) and best_score >= 0.95:
            print(f"Early stop: no improvement for {no_improve} BO steps with best_score={best_score:.3f}")
            break

    save_agent_checkpoint(agent_prev, last_ckpt, extra={'iter': final_iter, 'best_score': best_score})

    result = {
        'run_dir': str(run_dir),
        'history': history,
        'best_center': best_center.tolist(),
        'best_score': float(best_score),
        'best_payload': best_payload,
        'best_checkpoint': str(best_ckpt),
        'last_checkpoint': str(last_ckpt),
        'history_csv': str(hist_csv),
        'history_json': str(hist_json),
    }
    return result

## 8) Smoke Test (Fast)

Runs the minimal acceptance test:
- `T=3`, `K=20`, `B_r=5`


In [ ]:
assert WARMSTART_CHECKPOINT.exists(), f'Missing warmstart checkpoint: {WARMSTART_CHECKPOINT}'

smoke_result = run_bo_doraemon(
    checkpoint_path=WARMSTART_CHECKPOINT,
    mu0=MU0,
    sigma0=SIGMA0,
    T=SMOKE_CFG['T'],
    K=SMOKE_CFG['K'],
    B_r=SMOKE_CFG['B_r'],
    B_r_extra=SMOKE_CFG['B_r_extra'],
    bounds=DR_BOUNDS,
    tau_steps=tau_steps,
    alpha_low=alpha_low,
    alpha_high=alpha_high,
    sigma_min=SIGMA_MIN,
    sigma_max=SIGMA_MAX,
    surrogate_eval_episodes=SMOKE_CFG['surrogate_eval_episodes'],
    sigma_expand_factor=sigma_expand,
    sigma_shrink_factor=sigma_shrink,
    sigma_update_interval=sigma_update_interval,
    surrogate_mode=surrogate_mode,
    bo_epsilon_reset=bo_epsilon_reset,
    contender_margin=contender_margin,
    early_stop_patience=early_stop_patience,
    target_params=TARGET_PARAMS,
    seed=SEED,
)

print('Smoke run_dir:', smoke_result['run_dir'])
print('Smoke best center:', smoke_result['best_center'])
print('Smoke best target solve-rate:', smoke_result['best_score'])

## 9) Prototype Run (Medium)

Uncomment to run a fuller budget experiment:
- `T=12`, `K=120`, `B_r=10`


In [ ]:
# Prototype run (medium budget)
# prototype_result = run_bo_doraemon(
#     checkpoint_path=WARMSTART_CHECKPOINT,
#     mu0=MU0_EDGE,
#     sigma0=SIGMA0,
#     T=PROTO_CFG['T'],
#     K=PROTO_CFG['K'],
#     B_r=PROTO_CFG['B_r'],
#     B_r_extra=PROTO_CFG['B_r_extra'],
#     bounds=DR_BOUNDS,
#     tau_steps=tau_steps,
#     alpha_low=alpha_low,
#     alpha_high=alpha_high,
#     sigma_min=SIGMA_MIN,
#     sigma_max=SIGMA_MAX,
#     surrogate_eval_episodes=PROTO_CFG['surrogate_eval_episodes'],
#     sigma_expand_factor=sigma_expand,
#     sigma_shrink_factor=sigma_shrink,
#     sigma_update_interval=sigma_update_interval,
#     surrogate_mode=surrogate_mode,
#     bo_epsilon_reset=bo_epsilon_reset,
#     contender_margin=contender_margin,
#     early_stop_patience=early_stop_patience,
#     target_params=TARGET_PARAMS,
#     seed=SEED,
# )

# Hard-target run near deterministic limit (focused DR bounds)
# edge_result = run_bo_doraemon(
#     checkpoint_path=WARMSTART_CHECKPOINT,
#     mu0=MU0_EDGE,
#     sigma0=SIGMA0,
#     T=EDGE_CFG['T'],
#     K=EDGE_CFG['K'],
#     B_r=EDGE_CFG['B_r'],
#     B_r_extra=EDGE_CFG['B_r_extra'],
#     bounds=EDGE_DR_BOUNDS,
#     tau_steps=tau_steps,
#     alpha_low=alpha_low,
#     alpha_high=alpha_high,
#     sigma_min=SIGMA_MIN,
#     sigma_max=SIGMA_MAX,
#     surrogate_eval_episodes=EDGE_CFG['surrogate_eval_episodes'],
#     sigma_expand_factor=sigma_expand,
#     sigma_shrink_factor=sigma_shrink,
#     sigma_update_interval=sigma_update_interval,
#     surrogate_mode=surrogate_mode,
#     bo_epsilon_reset=bo_epsilon_reset,
#     contender_margin=contender_margin,
#     early_stop_patience=early_stop_patience,
#     target_params=TARGET_PARAMS,
#     seed=SEED,
# )

## 10) Visualization: BO Progress and Center Trajectory

In [ ]:
# choose which result to visualize
result = smoke_result  # or prototype_result
hist = result['history']

iters = [r['iter'] for r in hist]
fvals = [r['target_solve_rate'] for r in hist]
mu_f = [r['mu_force'] for r in hist]
mu_g = [r['mu_gravity'] for r in hist]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(iters, fvals, marker='o')
axes[0].set_title('TinySim target solve-rate vs BO iteration')
axes[0].set_xlabel('BO iteration')
axes[0].set_ylabel('Target solve-rate')
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(True, alpha=0.3)

sc = axes[1].scatter(mu_f, mu_g, c=iters, cmap='viridis', s=60)
axes[1].plot(mu_f, mu_g, alpha=0.5)
axes[1].set_title('Center trajectory in (force, gravity)')
axes[1].set_xlabel('force center')
axes[1].set_ylabel('gravity center')
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[1], label='BO iter')

plt.tight_layout()
plt.show()


## 11) Playback Best Policy in TinySim (Inline Animation)

In [ ]:
def render_tinysim_state_frame(position, min_position=-1.2, max_position=0.6, goal_position=0.5):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=100)
    x = np.linspace(min_position, max_position, 500)
    y = np.sin(3 * x)
    car_y = np.sin(3 * position)

    ax.plot(x, y, color='black', linewidth=2)
    ax.scatter([position], [car_y], s=200, c='crimson', zorder=5)

    gx = goal_position
    gy = np.sin(3 * gx)
    ax.plot([gx, gx], [gy, gy + 0.2], color='black', linewidth=2)
    ax.fill([gx, gx + 0.06, gx], [gy + 0.2, gy + 0.17, gy + 0.14], color='gold')

    ax.set_xlim(min_position - 0.05, max_position + 0.05)
    ax.set_ylim(-1.25, 1.25)
    ax.set_title('Best BO+DORAEMON Policy in TinySim')
    ax.set_xlabel('Position')
    ax.set_ylabel('Height')
    ax.grid(alpha=0.3)

    from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
    canvas = FigureCanvas(fig)
    canvas.draw()
    frame = np.asarray(canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return frame


def load_agent_from_checkpoint(path: str | Path, device: str = DEVICE) -> DQNAgent:
    payload = torch.load(str(path), map_location=device)
    agent = make_agent_from_hparams(payload.get('hparams'), device=device)
    agent.load_payload(payload)
    return agent


def record_best_policy_frames(agent: DQNAgent, target_params: dict, max_steps: int = 200):
    sim_env = MountainCarEnv()
    sim_env.force = float(target_params['force'])
    sim_env.gravity = float(target_params['gravity'])

    state = sim_env.reset()
    obs = np.array([state['position'], state['velocity']], dtype=np.float32)

    frames = [
        render_tinysim_state_frame(
            position=float(state['position']),
            min_position=sim_env.min_position,
            max_position=sim_env.max_position,
            goal_position=sim_env.goal_position,
        )
    ]

    done = bool(state['done'])
    steps = 0

    while (not done) and steps < max_steps:
        a = agent.act(obs, training=False)
        state = sim_env.step(a)
        obs = np.array([state['position'], state['velocity']], dtype=np.float32)
        done = bool(state['done'])
        steps += 1

        frames.append(
            render_tinysim_state_frame(
                position=float(state['position']),
                min_position=sim_env.min_position,
                max_position=sim_env.max_position,
                goal_position=sim_env.goal_position,
            )
        )

    summary = {
        'solved': bool(done),
        'steps': int(steps),
        'final_position': float(state['position']),
        'n_frames': len(frames),
    }
    return frames, summary


best_agent = load_agent_from_checkpoint(result['best_checkpoint'], device=DEVICE)
frames, summary = record_best_policy_frames(best_agent, target_params=TARGET_PARAMS, max_steps=tau_steps)

fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return [img]

ani = animation.FuncAnimation(fig, update, frames=frames, interval=25, blit=True)
plt.close(fig)

print(summary)
HTML(ani.to_jshtml())
